## 2.1 任务导向的 HuggingFace
接下来我们的目标是用 Hugging Face Transformers，完整、稳定、高效地跑通因果语言模型（数学推理）的 SFT/RL 训练全流程。

Transformer 本身只是一个通用的「骨架」，它本身不具备任何任务属性，真正决定模型能做什么的，是骨架顶层接的「任务头（Head）」。

- 接一个「语言模型头（LM Head）」—— 也就是一个把隐藏层特征映射到词表的线性层，它就能做自回归生成，也就是我们常说的大语言模型；
- 接一个「分类头」—— 把隐藏层特征映射到几个分类标签，它就能做情感分析、违禁内容检测；
- 接一个「序列到序列头」，它就能做翻译、摘要。

Hugging Face 没有让我们自己去拆骨架、改头，而是直接把「骨架 + 对应任务的头」打包成了一套按任务分类的 API，也就是`AutoModelForXXX`系列：
- `AutoModelForCausalLM`：对应「自回归生成任务」，也就是 GPT、Llama、Qwen 这类 decoder-only 模型，核心是「预测下一个 token」，也是我们做数学推理、对话模型的核心类；
- `AutoModelForSeq2SeqLM`：对应「输入到输出的转换任务」，比如翻译、摘要，是 encoder-decoder 结构的模型用的；
- `AutoModelForSequenceClassification`：对应「整句话分类」，比如情感分析；
- `AutoModelForTokenClassification`：对应「每个 token 打标签」，比如命名实体识别。

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "D:/mylab/cs336/Qwen2.5-Math-1.5B"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    # BF16 有 8 位指数，7 位尾数，与此同时 FP32 则是 8 对 23，指数位一致，
    # 也就是说能够表达的数值范围是完全一致的，仅仅只是精度（由尾数决定）会受到影响。
    # 但是 FP16（5 对 10）就会有溢出风险。
    low_cpu_mem_usage=True,
    attn_implementation="flash_attention_2",
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_path)

d:\Anaconda\envs\py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


### 2.1.1 加载预训练模型
`from_pretrained()` 先加载config.json，搞清楚模型的骨架结构（多少层、多少头、词表多大），根据配置，在内存里搭建一个空的模型结构，不占用额外内存；最后把safetensors里的预训练权重，精准地灌到骨架对应的每一层里。

原来 HF 的默认加载逻辑是先在 CPU 内存里，完整创建一个随机初始化的模型，再把预训练权重读进 CPU 内存，然后把权重灌到模型里，最后再把整个模型搬到 GPU 上。
这意味着，加载一个 7B BF16 的模型，CPU 内存至少要留 28GB 以上（两份模型的大小），70B 的模型直接需要 150GB 以上。

开启`low_cpu_mem_usage=True`之后，HF 会用`accelerate`库的能力，先在「元设备（meta device）」上搭一个只有结构、没有权重的「空壳模型」，完全不占 CPU 内存，然后直接把权重分片加载到 GPU 上。

**Flash Attention 2** 没有改变 Attention 的理论计算复杂度，它依然是 O (N²) 的复杂度，它的核心优化是 **工程层面的 IO 优化**。
原生 Attention 的痛点是：计算过程中，需要频繁地把中间数据在 GPU 的高速缓存（SRAM）和显存（HBM）之间来回搬运，IO 开销极大，尤其是序列长度变长的时候，速度会暴跌，显存占用也会暴涨。
Flash Attention 2 的核心，是把 Attention 的计算拆成了小块，让所有计算都在 SRAM 里完成，极大减少了 IO 次数（还有一些 work partitioning、warp/block 并行和非 matmul FLOPs 的优化），同时优化了 GPU 的并行效率。
现在 HF 已经把 Attention 后端做成了可切换的统一接口，除了 FA2，还有 SDPA（PyTorch 自带的）、Flex Attention，不用再改模型代码，直接换参数就能切换。
| backend名称 | 适用场景 | 你的场景适配性 |
| :--- | :--- | :--- |
| `eager` | PyTorch原生实现，无优化，兼容性最好 | 仅用于debug，不推荐训练用 |
| `sdpa` | PyTorch内置的缩放点积注意力，自动选最优内核 | 显卡不支持FA2时的首选，兼容性好 |
| `flash_attention_2` | 长序列训练最优，速度最快、显存最省 | 训练首选 |
| `flex_attention` | 自定义注意力模式（稀疏、滑动窗口） | 数学推理场景用不到 |

```python
# 训练用FA2，推理切SDPA，一行搞定
model.set_attn_implementation("sdpa")
```

### 2.1.2 前向传播与损失计算
`torch.nn.functional` 是PyTorch官方的神经网络函数库，里面包含了所有常用的无状态神经网络操作（比如交叉熵损失`cross_entropy`、Softmax、卷积等），都是纯函数式调用，不需要提前初始化层。

HF 原生推荐写法：

In [ ]:
# batch = {k: v.to("cuda") for k, v in train_batch.items()}

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],     # prompt/padding位置提前置为-100
)
loss = outputs.loss

传入 labels 的时候，`AutoModelForCausalLM` 会在模型内部自动完成两件事：
- 自动做「shift 错位」：把 input_ids 和 labels 错开一个 token，实现「预测下一个 token」的目标；
- 自动处理 loss 掩码：labels 里值为-100的位置，会自动被忽略，不参与 Loss 计算，正好对应我们的 prompt 部分、padding 部分。

当然我们也可以自己去移位和掩码：

In [ ]:
import torch.nn.functional as F
# PyTorch 的张量在内存里是按连续的顺序存储的，切片、转置这类操作，只会改变张量的 “视图”
# （也就是你看到的形状），不会改变内存里的实际存储顺序
# 如果后续操作（比如view()、cross_entropy）需要内存连续的张量，就会直接报错
# contiguous()会把张量在内存里重新排列成连续的顺序，返回一个全新的张量
# 切片操作之后，调用view()展平张量之前，必须加contiguous()

# input_ids形状假定[2, 128]
logits = outputs.logits[:, :-1, :].contiguous()     # 取前N-1个token的logits
                                                    # :-1表示 “从第 0 列到倒数第 2 列，不包含最后一列
                                                    # 对应 Causal LM 的输入，去掉最后一个 token，因为它没有后续目标可以预测
labels = batch["labels"][:, 1:].contiguous()        # 取后N-1个token当标签
                                                    # 逗号前面的:表示所有 batch 样本，1:表示 “从第 1 列到最后一列
                                                    # 对应 Causal LM 的标签，去掉第一个 token，因为它是初始输入，不是预测目标
loss = F.cross_entropy(
    logits.view(-1, logits.size(-1))                # 展平成[B * S, v]
    labels.view(-1)                                 # 展平成[B*S]
    ignore_index=-100,
    # 正常 token id 从 0 开始
    # 较短序列会被填充特殊 token（比如 [PAD]），这些位置的标签通常被设为 -100
    # 这样模型就不会因为预测这些无意义的填充位置而受到惩罚
)

### 2.1.3 保存分词器
保存 tokenizer 不是为了“保存训练后的变化”，而是为了“打包一个完整可用的模型包”，方便以后加载时能原样复原训练时的数据处理环境。

因为将来加载 checkpoint 的人（或未来的我们），不一定知道这个模型最初用的是哪个 tokenizer。如果只保存了模型权重，直接 from_pretrained(ckpt_dir) 会失败，因为 HuggingFace 需要 `tokenizer_config.json`、`vocab.json` 等文件来重建 tokenizer。

In [ ]:
output_dir = "./checkpoints"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

## 2.2 Huggingface 模型目录
以 Qwen2.5-Math-1.5B 为例：
```
Qwen2.5-Math-1.5B/
├── .hfd/
├── .gitattributes
├── config.json
├── generation_config.json
├── LICENSE
├── merges.txt
├── model.safetensors
├── README.md
├── tokenizer.json
├── tokenizer_config.json
└── vocab.json
```
一个一个来。
### 2.2.1 `config.json`
`config.json`里存的是模型的结构元信息：
```json
{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 4096,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_theta": 10000,
  "sliding_window": 4096,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.44.0",
  "use_cache": true,
  "use_mrope": false,
  "use_sliding_window": false,
  "vocab_size": 151936
}
```
| 字段 | 含义 | 值 |
| :--- | :--- | :--- |
| `architectures` | 模型对应的Python类名 | `Qwen2ForCausalLM`，AutoModel会根据这个字段选对应的类 |
| `hidden_size` | 每一层特征向量的维度 | 1536，维度越高，模型表达能力越强 |
| `num_hidden_layers` | Transformer Block的层数 | 28，模型的深度 |
| `num_attention_heads` | 多头注意力的头数 | 12 |
| `intermediate_size` | MLP层的中间维度 | 8960 |
| `max_position_embeddings` | 最大上下文窗口 | 32768，决定了最长能处理的CoT长度 |
| `vocab_size` | 词表大小 | 151936，模型能识别的token总数 |
| `rope_theta` | RoPE位置编码的底数 | 1000000.0，支持超长文本不乱码 |
| `torch_dtype` | 推荐的精度 | `bfloat16` |

> `architectures`字段只是记录了checkpoint对应的类名，但Auto类的解析逻辑，本质上是由配置与框架registry共同决定的，不是单靠这一个字段“拍板”。

### 2.2.2 `generate_config.json`
**这个文件里的参数，只影响推理阶段的`generate()`方法，不影响训练阶段的前向传播**。
```json
{
  "bos_token_id": 151643,             # 序列开始 token id
  "do_sample": false,                 # 是否采样（false 表示贪婪解码）
  "eos_token_id": 151643,             # 序列结束 token id
  "max_new_tokens": 2048,
  "transformers_version": "4.37.0"
}
```
### 2.2.3 `tokenizer_config.json`
注意！这里没有分词器的那些权重和信息，而是定义分词器行为的配置规则。
```json
{
  // 1️⃣ 基础行为
  "add_bos_token": false,
  // 是否自动在输入序列开头添加 BOS（begin of sequence）token。
  // false 表示不添加，因为 Qwen 模型通常由用户或模板自行控制。

  "add_prefix_space": false,
  // 是否在分词时自动在单词前加空格（某些 BPE tokenizer 需要）。

  // 2️⃣ 特殊 token 的完整定义（id → 属性）
  "added_tokens_decoder": {
    "151643": {
      "content": "<|endoftext|>",   // token 实际字符串
      "lstrip": false,              // 是否去除左侧空格
      "normalized": false,          // 是否经过 Unicode NFKC 归一化
      "rstrip": false,              // 是否去除右侧空格
      "single_word": false,         // 是否只能作为独立单词（不能与其他 token 合并）
      "special": true               // 是否为特殊 token（一般不会被拆分）
    },
    "151644": { "content": "<|im_start|>", ... },   // 消息开始标记
    "151645": { "content": "<|im_end|>", ... },     // 消息结束标记
    // ... 其他视觉/工具/代码补全专用 token
    "151657": { "content": "<tool_call>", "special": false },   // 非特殊 token，允许部分拆分
    "151658": { "content": "</tool_call>", ... }
  },

  // 3️⃣ 额外特殊 token 列表（用于 chat_template 或手动添加）
  "additional_special_tokens": [
    "<|im_start|>",
    "<|im_end|>",
    "<|object_ref_start|>",
    "<|object_ref_end|>",
    "<|box_start|>",
    "<|box_end|>",
    "<|quad_start|>",
    "<|quad_end|>",
    "<|vision_start|>",
    "<|vision_end|>",
    "<|vision_pad|>",
    "<|image_pad|>",
    "<|video_pad|>"
  ],
  // 这些 token 在分词时会作为一个整体，不会被 BPE 进一步拆分。

  // 4️⃣ 标准 token 别名
  "bos_token": null,
  // 未定义 BOS token（因为 add_bos_token=false，不需要）

  "eos_token": "<|endoftext|>",
  // 序列结束 token，对应 id 151643

  "pad_token": "<|endoftext|>",
  // 填充 token（与 eos_token 相同，Qwen 没有独立的 pad_token）

  "unk_token": null,
  // 未定义 UNK token（遇到未知 token 会直接报错，而不是替换为 <unk>）

  // 5️⃣ 对话模板（Jinja2 格式）
  "chat_template": "{%- if tools %}\n    {{- '<|im_start|>system\\n' }}\n    {%- if messages[0]['role'] == 'system' %}\n        {{- messages[0]['content'] }}\n    {%- else %}\n        {{- 'Please reason step by step, and put your final answer within \\\\boxed{}.' }}\n    {%- endif %}\n    {{- \"\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0]['role'] == 'system' %}\n        {{- '<|im_start|>system\\n' + messages[0]['content'] + '<|im_end|>\\n' }}\n    {%- else %}\n        {{- '<|im_start|>system\\nPlease reason step by step, and put your final answer within \\\\boxed{}.<|im_end|>\\n' }}\n    {%- endif %}\n{%- endif %}\n{%- for message in messages %}\n    {%- if (message.role == \"user\") or (message.role == \"system\" and not loop.first) or (message.role == \"assistant\" and not message.tool_calls) %}\n        {{- '<|im_start|>' + message.role + '\\n' + message.content + '<|im_end|>' + '\\n' }}\n    {%- elif message.role == \"assistant\" %}\n        {{- '<|im_start|>' + message.role }}\n        {%- if message.content %}\n            {{- '\\n' + message.content }}\n        {%- endif %}\n        {%- for tool_call in message.tool_calls %}\n            {%- if tool_call.function is defined %}\n                {%- set tool_call = tool_call.function %}\n            {%- endif %}\n            {{- '\\n<tool_call>\\n{\"name\": \"' }}\n            {{- tool_call.name }}\n            {{- '\", \"arguments\": ' }}\n            {{- tool_call.arguments | tojson }}\n            {{- '}\\n</tool_call>' }}\n        {%- endfor %}\n        {{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}\n        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != \"tool\") %}\n            {{- '<|im_start|>user' }}\n        {%- endif %}\n        {{- '\\n<tool_response>\\n' }}\n        {{- message.content }}\n        {{- '\\n</tool_response>' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}\n            {{- '<|im_end|>\\n' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- '<|im_start|>assistant\\n' }}\n{%- endif %}\n",
  // 定义了如何将 messages 列表转换成模型输入的字符串。
  // 包含对 system/user/assistant/tool 角色、工具调用、视觉输入等的格式化规则。
  // 当你调用 tokenizer.apply_chat_template() 时，就会使用这个模板。

  // 6️⃣ 其他配置
  "clean_up_tokenization_spaces": false,
  // 解码时是否清理多余空格（false 表示保留原始空格分布）

  "errors": "replace",
  // 解码出错时的处理方式，"replace" 表示用 � 替换无效字节

  "model_max_length": 131072,
  // 模型最大上下文长度（128K tokens），用于自动截断

  "split_special_tokens": false,
  // 是否允许特殊 token 被拆分（false 表示保持完整）

  "tokenizer_class": "Qwen2Tokenizer",
  // 指定 tokenizer 的类名，transformers 会据此动态加载正确的实现
}
```
chat 模型收到的并不是 **天然的消息列表**，而是 **消息列表先经过 tokenizer 的 `apply_chat_template()`，被转成一串带控制token的文本/序列**。不同模型就算基座相同，也可能要求完全不同的控制token，不能用错模板。

比如之前我们预处理了GSM8K数据集，prompt已经固定的格式的解析、特殊token的处理，全都是由`tokenizer_config.json`里的`chat_template`字段定义的。它本质上是一段Jinja2模板代码，定义了多轮对话、用户/助手消息、系统提示词怎么拼接成最终的文本。

### 2.2.4 `tokenizer.config`
`vocab.json`/`merges.txt`/`tokenizer.json` 是之前A1学习的时候，BPE分词器的字节对编码的词表和合并规则相关配置，可以回去查看。
### 2.2.5 `safetensor`
超过10GB的大模型权重会自动分片保存，分成多个`.safetensors`文件，同时生成一个`index.json`索引文件。这里我们只有大概3GB左右的大小所以没有`model.safetensors.index.json`。不过一般这个里面会有：
- `metadata`字段：存了模型的总大小等；
- `weight_map`字段：一个字典，把每一层的参数名，映射到它所在的分片文件里，比如`model.layers.0.self_attn.q_proj.weight`在`model-00001-of-00002.safetensors`里。

相比传统基于`pickle`的`.bin`格式，`safetensors`有两个核心优势：
1. **安全性**：`.bin`文件可能包含恶意代码，而`safetensors`只存纯数据，不会执行任何代码；
2. **速度**：支持内存映射（Zero-copy），加载速度比`.bin`快很多。

## 2.3 因果语言模型训练数据
### 2.3.1 三种掩码
训练时，我们会把一句话（比如 “I love NLP”）切成 [I, love, NLP]，然后让模型在每个位置预测下一个 token。所以第一种 **因果掩码（Causal Mask）** 我们已经很熟悉了，限制注意力只能看到当前及之前的 token。
这个掩码是模型架构自带的，**不需要手动构造**，transformers 在 forward 时会自动生成。

现在我们有了 GSM8K 数据，每条样本包含 prompt（数学题）和 response（解答过程）。
你希望模型学会：给定 prompt，生成正确的 response。

但在训练时，我们实际上是把 prompt + response 拼接成一条完整序列喂给模型，按照“预测下一个 token”的规则，模型会尝试预测每一个 token——包括 prompt 内部的 token。

因为我们不可能让模型把用户的数学题也背下来。所以 **prompt 和 response 都进了模型，但我们通常只让 response 参与 loss 计算**。

这就是 **loss mask（损失掩码）** 的由来——在构造 labels 时，把 prompt 对应位置设为 **-100**，交叉熵自动忽略它们。

最后是一个在我系统学习之前听得很多的知识点，一个 batch 里序列长度不同，我们需要把短的序列用 `[PAD]` token 填充到相同长度。
但这些填充的 token 是没有任何意义的，模型不应该注意它们，也不应该让它们的预测产生损失。解决方法是：
- 在注意力计算中：通过 `attention_mask` 让填充位置的注意力分数为 0（相当于这些 token 是“空气”）。
- 在损失计算中：同样把填充位置的 labels 设为 -100（和 prompt 部分一样处理）。
> （Causal LM 必须右填充）

In [ ]:
from typing import List, Dict
import torch
from transformers import PreTrainedTokenizer

IGNORE_INDEX = -100

def build_gsm8k_sft_batch(
    prompt_strs: List[str],
    output_strs: List[str],
    tokenizer: PreTrainedTokenizer,
    max_seq_len: int = 1024
) -> Dict[str, torch.Tensor]:
    input_id_list = []
    label_list = []

    pad_id = tokenizer.pad_token

    # 遍历每条数据，分词、拼接、构造label
    for prompt, output in zip(prompt_strs, output_strs):
        # 分别对prompt和response分词，不自动加特殊token，我们手动控制
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False, truncation=True, max_length=max_seq_len)
        output_ids = tokenizer.encode(output, add_special_tokens=False, truncation=True, max_length=max_seq_len)

        # 在response后面硬塞一个eos token id
        output_ids = output_ids + [tokenizer.eos_token_id]

        input_ids = prompt_ids + output_ids
        # 初始化labels，使其与input_ids完全一致
        labels = input_ids.copy()

        # 损失掩码
        labels[:len(prompt_ids)] = [IGNORE_INDEX] * len(prompt_ids)

        input_id_list.append(input_ids)
        label_list.append(labels)

    # padding
    # 如果 actual_max_len 超过了 max_seq_len，我们不能按照 actual_max_len 做 padding
    # 所以最外层是个min作裁剪
    max_len = min(max(len(x) for x in input_id_list), max_seq_len)
    batch_size = len(input_id_list)

    # [p1, p2, r1, r2, eos]，让input_ids补齐<PAD>
    batch_input_ids = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    # [-100, -100, r1, r2, eos]，现在还对的齐，forward后labels就会右移一位
    batch_lables = torch.full((batch_size, max_len), IGNORE_INDEX, dtype=torch.long)

    # attention mask，有效token为1，padding为0
    batch_attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)

    # 上面的三个张量都是空白。下面才是开始填数据
    # 把每个样本的真实数据“贴”到之前铺好的全填充张量里
    for i, (ids, labels) in enumerate(zip(input_id_list, label_list)):
        seq_len = min(len(ids), max_len)

        # i 是样本的序号（0 或 1）—— 选哪一行
        # :seq_len 是列范围：:3 表示“从第 0 列开始，取到第 2 列（不包含第3列）
        # 比如说假定batch_input_ids[0, :3] 
        # 原来 batch_input_ids[0] = [pad, pad, pad, pad]
        # 赋值后：                [101, 102, 103, pad]
        batch_input_ids[i, :seq_len] = torch.tensor(ids[:seq_len], dtype=torch.long)

        batch_labels[i, :seq_len] = torch.tensor(lables[:seq_len], dtype=torch.long)

        # 原来 batch_attention_mask[0] = [0, 0, 0, 0]
        # 赋值后：                       [1, 1, 1, 0]
        batch_attention_mask[i, :seq_len] = 1

    return {
        "input_ids": batch_input_ids,
        "labels": batch_lables,
        "attention_mask": batch_attention_mask,
    }

### 2.3.2 形状流水线
- 单条数据
```
prompt: "A conversation between User and Assistant...\nUser: 题目\nAssistant: \n"
response: "\nNatalia sold 48/2 = 24 clips in May.\n...\n\n\n72\n"
```
- 分词后
```
prompt_ids:   [p1, p2, p3, ..., pn] （长度30）
response_ids: [r1, r2, r3, ..., rm, eos] （长度80）
```
- 拼接后（无padding）
```
input_ids:    [p1, p2, ..., p30, r1, r2, ..., r80, eos] （总长度111）
labels:       [-100, -100, ..., -100, r1, r2, ..., r80, eos] （prompt部分全是-100）
attention_mask: [1, 1, ..., 1, 1, 1, ..., 1, 1] （全是1，无padding）
```
- padding后（假设补到长度128，右填充）
```
input_ids:    [p1-p30, r1-r80, eos, pad, pad, ..., pad] （补17个pad）
labels:       [-100*30, r1-r80, eos, -100, -100, ..., -100] （padding部分全是-100）
attn_mask:    [1*111, 0, 0, ..., 0] （有效位置1，padding位置0）
```

### 2.3.4 padding-free packing
常规方法中，为了将不同长度的序列组合成一个批量 batch，需要对短序列进行填充。这些 `[PAD]` token 对应的计算完全是浪费的，在最大序列长度远大于平均长度的数据集上尤为显著。

本质上，padding 用一个简单的数学手段（填充张量）绕过了上层问题（数据不等长），却在底层（GPU计算）付出了巨大的算力代价。

因此，padding-free / packing 技术应运而生，成为提高训练吞吐量的关键优化。

**序列打包** 核心思想是把一个批次里的多个样本 packing（打包） 在一起，串联成一个超长序列，完全移除填充 token。该技术被称为 packing，而它在 Hugging Face 生态中的一种具体状态被称为 padding_free。

传统模式下，len_seq 个序列会填充到和最长者一样长（添加 `[PAD]`）。打包模式则将它们无缝拼接，可以"追尾"式串联，通过注意力掩码确保 token 只在自己的原始序列内部互动，避免跨样本污染。拼接后生成一个超长序列，同时生成一个特定的 attention_mask 或用 cu_seqlens 记录边界，确保注意力不会跨序列流动。

```
   [ 0, 0, 0, P, P, P ] (填充)
   [ 1, 1, 1, 1, 1, 1 ]
→  [ 2, 2, P, P, P, P ]
   [ 3, 3, 3, P, P, P ]

   [ 0, 0, 0 ]
→  [ 1, 1, 1, 1, 1 ]
   [ 2, 2 ]
   [ 3, 3, 3, P ] (最后可能有一点填充)
```
| 场景 | 配置方法 | 说明 |
| :--- | :--- | :--- |
| **原生 `transformers` Trainer** | **`DataCollatorWithFlattening`** | 自动完成序列拼接，是官方最推荐的打包数据整理器（Data Collator） |
| **TRL `SFTTrainer` (旧方法)** | `DataCollatorForCompletionOnlyLM(padding_free=True)` | 在特定的 `DataCollatorForCompletionOnlyLM` 中启用 `padding_free=True` |
| **TRL `SFTTrainer` (新方法)** | `completion_only_loss=True` + 标准化数据集 | 在 `SFTConfig` 中开启 `completion_only_loss=True`，配合 format 写好的 prompt-completion 数据集 |

虽然 packing 能显著提升效率，但实现复杂。因此工业界在多模型、多架构场景下常采用更简洁的方案，例如 AI21 提出的 padding-aware micro-batching，将长序列和短序列分开打包，进一步减少填充浪费。这类方法与 packing 并列但互斥，不用特殊 attention_mask，适合混合架构（如 Mamba）。

## 2.4 熵监控
熵就是不确定性。这个之前投机解码还是哪儿是学到过这个概念的，直观来说，`next-token entropy` 是模型在当前位置，对下一个该输出什么 token，到底有多拿不准。
- **高熵**：模型觉得很多个 token 的概率都差不多，非常迷茫，不知道该选哪个 —— 比如训练初期，模型根本不知道怎么解题，就会处于高熵状态；
- **低熵**：模型几乎把全部概率都压到某一个 token 上，非常确定自己该输出什么 —— 比如模型学会了解题步骤，就会处于低熵状态。

一旦奖励设计有漏洞，模型很容易走向 **mode collapse（模式坍缩）**：它会发现只要反复输出少数几种 “稳妥高分的套话”，就能稳定拿到高奖励，看起来会答题，实际上已经在偷懒，彻底失去了泛化能力。
而模式坍缩最直接最敏感的信号，就是熵会突然暴跌到接近 0，这时候光看 Loss 可能还在降，但模型已经废了。
### 2.4.1 公式推导
$$
H(P) = - \sum_i p_i \log p_i
$$
把 $p_i = \mathrm{softmax}(z_i)$ 代进去：
$$
H(P) = \mathrm{LSE}(z) - \sum_i p_i z_i
$$
- LSE就是Log-Sum-Exp，`torch.logsumexp`，A1 应该看过。
- $$\text{LSE} = \log\left(\sum e^{\text{logits}_i}\right)$$
- 避免了对极小概率直接做log()，不会出现 log(0) = NaN，在数学上等价于 log(softmax(x))，但是log_softmax是用替代公式去算，更快更稳定。

### 2.4.2 全局熵与响应熵
- **global entropy（全局熵）**：计算整条序列（prompt+response）的平均不确定性，它会被固定的 prompt 模板稀释，只能反映模型对 “这类题目整体分布” 的熟悉程度，参考价值有限；
- **response entropy（响应熵）**：只计算模型生成的 response 部分（推理过程 + 答案）的平均不确定性，它完全反映了模型在真正解题时的确定性，是必须重点监控的核心指标。
### 2.4.3 对数概率
熵是衡量模型“不确定性”的指标，而对数概率（log probability）是衡量模型对某个具体 token 预测得“有多确定”的指标。

在因果语言模型的训练和评估中，我们经常需要回答一个问题：对于给定的正确答案 token，模型认为它应该出现的“对数可能性”是多少？

1. **计算交叉熵损失**：
`loss = -log_probs[正确token]`
交叉熵本质上就是 **负的对数概率**。训练时最小化 loss = 最大化正确 token 的 log_probs。

2. **评估 ppl（困惑度）**：
`ppl = exp(-mean(log_probs))`
需要整个序列所有 token 的 log_probs 的平均值。

3. **强化学习（RLHF）中的奖励基准**：
旧策略和新策略的 KL 散度需要对比 log_probs。

4. **检查模型是否过度自信**：
正确 token 的 log_probs 很大（接近 0），说明模型很确定；如果很小（负很大），说明模型对这个正确答案没信心。

我们的 GSM8K SFT 训练中，主要就是通过交叉熵 loss 来更新模型，而交叉熵直接依赖对数概率。

In [ ]:
def compute_entropy(logits: torch.Tensor) -> torch.Tensor:
    lse = torch.logsumexp(logits, dim=-1)       # [B, S]

    probs = F.softmax(logits, dim=-1)           # [B, S, V]

    exp_logits = torch.sum(probs * logits, dim=-1)  # [B, S]

    return lse - exp_logits


def get_log_probs_and_entropy(
    model,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    labels: torch.Tensor=None,
    reponse_mask: torch.Tensor=None,
):
    # forward
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    # 计算逐token熵
    token_entropy = compute_entropy(logits)

    # 计算所有 token 在所有词表上的 log_prob，形状 [B, S, V]
    log_probs_all = F.log_softmax(logits, dim=-1)

    result = {
        "logits": logits,
        "token_entropy": token_entropy,
        "avg_global_entropy": token_entropy[attention_mask == 1].mean().item()
    } 

    # 如果有的话，计算Response里的平均熵
    if response_mask is not None:
        result["avg_res_entropy"] = token_entropy[response_mask == 1].mean().item()
    
    if labels is not None:
        # 从log_probs_all中提取labels位置对应的对数概率
        # gather(dim=-1, index=labels.unsqueeze(-1))
        # 对于每个 batch、每个位置，取 log_probs_all 在那个位置上的第 labels[b,s] 个值
        # 结果形状 [B, S, 1]，再 squeeze() 变为 [B, S]
        token_log_probs = torch.gather(
            log_probs_all,
            dim=-1,
            index=label.unsqueeze(-1).clamp_min(0)  # -100的labels被clamp为0
        )   # -100 直接用作 index 会越界（词表索引从 0 开始）

        # 只保留有效位置的log概率
        valid_mask = (labels != -100)
        token_log_probs = token_log_probs * valid_mask

        result["token_log_probs"] = token_log_probs
        result["valid_mask"] = valid_mask

        return result

## 2.5 显存优化
### 2.5.1 梯度累积
> 见A1

### 2.5.2 梯度检查点
梯度累积解决了 batch size 的问题，而梯度检查点解决了长序列训练时，中间激活值把显存占满的核心痛点。

训练大语言模型时，显存主要被三样东西吃掉：
| 显存占用项 | 典型大小（1.5B BF16 + AdamW） | 特点 |
|-----------|------------------------------|------|
| **模型参数** | 3 GB | 固定的 |
| **优化器状态**（AdamW 的动量和方差） | 12 GB | 固定的 |
| **中间激活值**（每一层的输出） | 与 `batch_size × seq_len × hidden_dim` 成正比，**线性增长** | 变化的，序列越长越大 |

#### 正常训练（不用检查点）
- **前向传播**：逐层计算，每层都保存该层的输入（或输出激活值）以备反向传播使用。  
  显存占用 = 所有层的激活值之和（随层数、序列长度线性增长）。
- **反向传播**：从最后一层开始，依次读取保存的激活值，计算梯度。

#### 开启梯度检查点
- **前向传播**：只保存 **少数几个检查点**（比如每 k 层保存一次，k 通常取 2、4 等）。  
  大部分中间激活值 **不保存**，直接丢弃。
- **反向传播**：当需要某一层的激活值来算梯度时，从最近的一个检查点开始，**重新执行那段的前向传播**，临时把那几层的激活值再算一遍，用完就丢弃。

```python 
# 一行开启，HF Transformers 原生支持
model.gradient_checkpointing_enable()

# 如果想更精细控制（PyTorch 原生 API）：
from torch.utils.checkpoint import checkpoint_sequential
# 或者使用 checkpoint 函数包装自定义层
```
#### 细节
1. 推荐 use_reentrant=False
PyTorch 的 torch.utils.checkpoint 函数有一个 use_reentrant 参数。目前默认是 True（向后兼容），但官方文档推荐设置为 False，因为功能更全、更安全，未来会成为默认值。

2. 小心 detach 的张量
如果你在 checkpoint 区域内部对张量执行 .detach()，反向传播可能无法正确计算梯度。尽量避免。

3. 随机数操作（如 dropout）
检查点会自动保存和恢复随机数生成器（RNG）状态，保证两次重计算的结果完全一致（确定性）。这会带来轻微的性能开销，但可以接受。
